# Disease Spread Modeling Using Dynamic Bayesian Networks

**PGM Course Project — End-to-End Notebook**

---

## Project Description

This project models the **temporal spread of infectious diseases through a population** using **Dynamic Bayesian Networks (DBNs)**.

| Concept | Implementation |
|---------|----------------|
| **Latent variables** | Individual SEIR states: Susceptible (S), Exposed (E), Infectious (I), Recovered (R) |
| **Observations** | Reported symptoms or **test results** (positive / negative / missing) |
| **Representation** | 2-time-slice DBN graph + Conditional Probability Tables (CPTs) |
| **Inference** | Forward–backward belief propagation |
| **Learning** | EM algorithm to estimate β, σ, γ from epidemic data |

### Key Query

> *Given a set of symptom/test observations across a network of individuals, what is the probability that a given node is currently infectious?*

$$P(X_i^t = \text{Infectious} \mid \text{all observations})$$

---

**Notebook roadmap:**
1. Data description & loading (real-world Geneva COVID-19 contact tracing)
2. Representation — DBN structure and CPTs
3. Inference — posterior beliefs and figures
4. Learning — EM parameter estimation
5. Summary and interpretation of all figures

## 0. Setup

Run this cell first. It imports project modules and configures plotting for inline display in the notebook.

In [ ]:
import sys
from pathlib import Path

# Find project root (works whether kernel cwd is project/ or notebooks/)
_cwd = Path.cwd()
if (_cwd / "src").is_dir():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "src").is_dir():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11

from src.config import ModelParams, SimConfig, STATES, STATE_IDX, OBS_MISSING, OBS_POS, OBS_NEG
from src.real_data import load_real_data, RAW_DIR, GENEVA_FILES
from src.model import build_dbn_structure, transition_distribution, emission_likelihood
from src.network import network_summary
from src.inference import infer_infectious_probability, query_node_infectious
from src.learning import em_learn
from src.simulation import epidemic_counts

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Figures will be saved to: {OUTPUT_DIR}')

---
# Part 1 — Data Description

## 1.1 What data does this project use?

The project supports **simulated or real-world epidemic data** (as required by the PGM project description).

**This notebook uses real-world data by default:**

| Item | Detail |
|------|--------|
| **Dataset** | Geneva COVID-19 contact tracing (GEgraph) |
| **Source** | [PersonalDataIO/GEgraph](https://github.com/PersonalDataIO/GEgraph) |
| **Location** | Canton of Geneva, Switzerland, 2020 |
| **Population** | De-identified individuals from an outbreak cluster |
| **Network** | Close-contact edges from tracing records |
| **Observations** | Dates of positive PCR tests |

### Mapping to the DBN model

| DBN concept | Real data |
|-------------|----------|
| Individual \(i\) | A person in the contact-tracing subgraph |
| Latent state \(X_i^t\) | Hidden SEIR state (not directly observed) |
| Observation \(Y_i^t\) | Positive test on day \(t\), or **missing** if no test |
| Contact edge | Two people were in close contact (transmission possible) |

### Observation encoding

| Value | Symbol | Meaning |
|-------|--------|--------|
| `1` | Positive | Reported positive test (symptom/test result) |
| `0` | Negative | Reported negative test |
| `-1` | Missing | No test or symptom report that day |

## 1.2 Inspect raw data files

The loader downloads two CSV files into `data/raw/` on first run.

In [ ]:
from src.real_data import _download_geneva_raw
_download_geneva_raw()

suivi = pd.read_csv(RAW_DIR / GENEVA_FILES['suivi'])
entourage = pd.read_csv(RAW_DIR / GENEVA_FILES['entourage'])

print('=== redcap_suivi.csv (follow-up & test dates) ===')
print(f'Rows: {len(suivi):,}')
display(suivi.head(3))

print('\n=== redcap_entourage.csv (contact links) ===')
print(f'Rows: {len(entourage):,}')
display(entourage.head(3))

## 1.3 Load the epidemic subgraph

We extract a **connected subgraph** of ~30 individuals from an early March 2020 outbreak cluster:

1. Pick an early infected person with several contacts (seed)
2. Expand by breadth-first search to collect neighbors
3. Build contact network and map positive test dates to daily timesteps

In [ ]:
config = SimConfig(n_nodes=30, data_source='real')
bundle = load_real_data(config)

G = bundle.graph
Y_obs = bundle.Y_obs
X_true = bundle.X_true
patient_zero = bundle.patient_zero
meta = bundle.metadata

print('Dataset:', bundle.dataset_name)
print('Source:', meta['source'])
print(f"People (nodes): {meta['n_nodes']}")
print(f"Time steps (days): {meta['n_timesteps']}")
print(f"Period: {meta['date_start']} → {meta['date_end']}")
print(f"Seed person (patient zero): ID {meta['seed_person_id']} → node index {patient_zero}")
print(f"Positive tests in subgraph: {meta['n_positive_tests']}")
print(f"Total observations matrix size: {Y_obs.shape} (time × people)")
print(f"Non-missing tests: {(Y_obs >= 0).sum()} / {Y_obs.size} ({100*(Y_obs>=0).mean():.1f}%)")

In [ ]:
# Show which individuals tested positive and when
obs_records = []
for t in range(Y_obs.shape[0]):
    for i in range(Y_obs.shape[1]):
        if Y_obs[t, i] == OBS_POS:
            obs_records.append({
                'node_index': i,
                'person_id': bundle.node_ids[i],
                'day_index': t,
                'calendar_date': str(bundle.dates[t].date()),
                'observation': 'POSITIVE',
            })

obs_df = pd.DataFrame(obs_records)
print('Positive test observations in our subgraph:')
display(obs_df)

---
# Part 2 — Representation (PGM Pillar 1)

## 2.1 Dynamic Bayesian Network structure

A **2-time-slice DBN** repeats the same structure at every timestep:

```
         t-1                    t
    X_i^{t-1} ──────────→ X_i^t  ───→ Y_i^t
         ↑                    ↑
    X_j^{t-1} ──────────→ X_j^t
    (contact neighbors j)
```

**Edges:**
- $X_i^{t-1} \to X_i^t$ — within-person SEIR dynamics
- $X_j^{t-1} \to X_i^t$ — transmission from infectious neighbors
- $X_i^t \to Y_i^t$ — symptom/test emission

**Latent states:** $X_i^t \in \{S, E, I, R\}$

In [ ]:
structure = build_dbn_structure(G)
summary = network_summary(G)

print(structure['description'])
print(f"\nNetwork statistics:")
print(f"  Nodes (individuals): {summary['n_nodes']}")
print(f"  Edges (contacts):    {summary['n_edges']}")
print(f"  Average degree:      {summary['avg_degree']:.2f}")
print(f"  Connected:           {summary['is_connected']}")

## 2.2 Conditional Probability Tables (CPTs)

The model is parameterized by three epidemiological rates:

| Parameter | Symbol | Meaning |
|-----------|--------|--------|
| Transmission | β (beta) | Probability of infection per infectious contact |
| Progression | σ (sigma) | Rate E → I (exposed becomes infectious) |
| Recovery | γ (gamma) | Rate I → R (infectious recovers) |

**Transmission (noisy-OR):** for a susceptible person $i$,

$$P(X_i^t = E \mid S) = 1 - \prod_{j \in \text{neighbors}} (1 - \beta \cdot P(X_j^{t-1} = I))$$

**Emission (test model):**
- Sensitivity = P(positive test | infectious)
- Specificity = P(negative test | not infectious)

In [ ]:
params = ModelParams(beta=0.30, sigma=0.20, gamma=0.10)

print('Example CPT: P(X^t | X^{t-1}=S, one infectious neighbor with P(I)=1.0)')
p = transition_distribution('S', [1.0], params)
for s, prob in zip(STATES, p):
    print(f'  P({s}) = {prob:.3f}')

print('\nEmission likelihood P(Y | X):')
for state in STATES:
    lik_pos = emission_likelihood(OBS_POS, params)
    lik_neg = emission_likelihood(OBS_NEG, params)
    print(f'  {state}: P(pos)={lik_pos[STATE_IDX[state]]:.2f}, P(neg)={lik_neg[STATE_IDX[state]]:.2f}')

---
## Figure 0 — Contact Network

**What it shows:** The population structure — who can infect whom through close contacts.

**How to read it:**
- Each **node** = one individual in the subgraph
- Each **edge** = a documented close contact from tracing records
- Disease can spread along edges from infectious to susceptible neighbors

**Why it matters:** The DBN couples each person's state to their neighbors' previous states. Network topology directly affects how fast and how far the epidemic spreads.

In [ ]:
pos = nx.spring_layout(G, seed=42)
fig, ax = plt.subplots(figsize=(8, 6))
nx.draw(G, pos, with_labels=True, node_color='#a8d4f0', node_size=500,
        font_size=9, edge_color='#888888', ax=ax)
ax.set_title('Figure 0: Contact Network (Real Geneva Tracing Subgraph)', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig0_network.png', dpi=150)
plt.show()

---
## Figure 1 — Epidemic Curve

**What it shows:** How many individuals are in each SEIR state over time.

**How to read it:**
- **Red (I):** infectious count — the epidemic "peak"
- **Orange dashed (E):** exposed but not yet infectious
- **Green dotted (R):** recovered

**Note for real data:** Latent states are *approximated* from test dates for visualization only. The model never sees these during inference — only partial test observations.

**Why it matters:** Confirms the subgraph captures a real outbreak wave with rise and fall of infections.

In [ ]:
counts = epidemic_counts(X_true)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(counts['I'], label='Infectious (I)', linewidth=2, color='#c0392b')
ax.plot(counts['E'], label='Exposed (E)', linestyle='--', color='#e67e22')
ax.plot(counts['R'], label='Recovered (R)', linestyle=':', color='#27ae60')
ax.set_xlabel('Time step (days from start of window)')
ax.set_ylabel('Number of individuals')
ax.set_title('Figure 1: Epidemic Curve (Approximate SEIR from Real Test Dates)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig1_epidemic_curve.png', dpi=150)
plt.show()

peak_t = counts['I'].index(max(counts['I']))
print(f'Peak infectious count: {max(counts["I"])} at day t={peak_t} ({bundle.dates[peak_t].date()})')

---
# Part 3 — Inference (PGM Pillar 2)

## 3.1 Forward–backward belief propagation

Inference computes the **posterior** over latent SEIR states given all observations:

**Forward pass (filtering):**
$$\alpha_t(i,s) = P(X_i^t = s \mid Y^{1:t})$$

1. **Predict** next state from SEIR transitions and neighbor beliefs
2. **Update** with test observation $Y_i^t$ (if not missing)

**Backward pass (smoothing):**
$$\gamma_t(i,s) = P(X_i^t = s \mid Y^{1:T}) \propto \alpha_t(i,s) \cdot \beta_t(i,s)$$

Uses **future** observations to refine past beliefs.

### The key query

$$P(X_i^t = I \mid \text{all } Y) = \gamma_t(i, I)$$

In [ ]:
P_I, beliefs = infer_infectious_probability(
    G, Y_obs, params, patient_zero=patient_zero, smooth=True,
)

print(f'Posterior P(I) shape: {P_I.shape} (time × people)')
print(f'Range: [{P_I.min():.3f}, {P_I.max():.3f}]')

---
## Figure 2 — Posterior Heatmap P(Infectious)

**What it shows:** For every person and every day, the model's belief that they are **infectious**.

**How to read it:**
- **X-axis:** time (days)
- **Y-axis:** individual (node index)
- **Color:** red = high P(infectious), white = low
- **Bright horizontal bands** often align with positive test dates
- **Missing tests** → inference relies on network propagation from neighbors

**Why it matters:** This is the central output of the DBN — recovering hidden infection states from sparse, noisy observations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(P_I.T, aspect='auto', origin='lower', cmap='Reds', vmin=0, vmax=1)
ax.set_xlabel('Time step (days)')
ax.set_ylabel('Individual (node index)')
ax.set_title('Figure 2: P(Infectious | All Observations)')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('P(I)')

# Mark positive test days with white dots
for t in range(Y_obs.shape[0]):
    for i in range(Y_obs.shape[1]):
        if Y_obs[t, i] == OBS_POS:
            ax.plot(t, i, 'wo', markersize=4, markeredgecolor='black', markeredgewidth=0.5)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig2_heatmap_P_I.png', dpi=150)
plt.show()
print('White dots = observed positive tests')

---
## Figure 3 — Network Colored by Posterior

**What it shows:** Each person colored by their **maximum** posterior infectious probability over all time.

**How to read it:**
- **Dark red nodes** = likely infected at some point during the outbreak
- **Pale nodes** = remained mostly susceptible
- **Patient zero** (seed) and highly connected individuals tend to be redder

**Why it matters:** Summarizes who the model believes was most affected — useful for identifying high-risk individuals in the contact network.

In [ ]:
max_P = P_I.max(axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
nodes = nx.draw_networkx_nodes(G, pos, node_color=max_P, cmap=plt.cm.Reds,
                               vmin=0, vmax=1, node_size=550, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.35, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
cbar = fig.colorbar(nodes, ax=ax)
cbar.set_label('max over time: P(I)')
ax.set_title('Figure 3: Network Colored by Posterior P(Infectious)')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig3_network_posterior.png', dpi=150)
plt.show()

## 3.2 Answering the project query

> *Given symptom/test observations across the network, what is P(node i is infectious at time t)?*

In [ ]:
# Example queries for several nodes and times
query_examples = [
    (patient_zero, 10),
    (patient_zero, min(30, Y_obs.shape[0]-1)),
    (1, 10),
    (5, 20),
]

rows = []
for node, t in query_examples:
    p = query_node_infectious(G, Y_obs, params, node=node, time=t, patient_zero=patient_zero)
    y_val = Y_obs[t, node]
    y_str = {OBS_POS: 'POSITIVE', OBS_NEG: 'NEGATIVE', OBS_MISSING: 'missing'}.get(y_val, str(y_val))
    approx_state = STATES[X_true[t, node]] if X_true is not None else 'N/A'
    rows.append({
        'node': node,
        'person_id': bundle.node_ids[node],
        'day_t': t,
        'date': str(bundle.dates[t].date()),
        'P(infectious)': round(p, 4),
        'observation': y_str,
        'approx_true_state': approx_state,
    })

display(pd.DataFrame(rows))

---
# Part 4 — Learning (PGM Pillar 3)

## 4.1 EM Algorithm

When epidemiological parameters (β, σ, γ) are **unknown**, we learn them from observations using **Expectation–Maximization**:

| Step | Action |
|------|--------|
| **E-step** | Run forward–backward inference → soft SEIR state assignments |
| **M-step** | Re-estimate β, σ, γ from expected transition counts |
| **Repeat** | Until parameters converge |

On **real data**, there is no ground-truth β, σ, γ — we learn whatever best explains the observed tests.

On **synthetic data**, we can compare learned vs. true parameters (see optional section below).

In [ ]:
init_params = ModelParams(beta=0.10, sigma=0.10, gamma=0.10)
print('Initial guess:', init_params)
print('\nRunning EM...')
learned, history = em_learn(G, Y_obs, init_params, n_iter=25, patient_zero=patient_zero, verbose=True)
print(f'\nLearned parameters: beta={learned.beta:.3f}, sigma={learned.sigma:.3f}, gamma={learned.gamma:.3f}')

---
## Figure 4 — EM Convergence

**What it shows:** How β, σ, and γ change across EM iterations.

**How to read it:**
- Each colored line = one parameter's learned value per iteration
- **Flat tail** = convergence
- On real data, dashed "true" lines are absent (unknown ground truth)

**Why it matters:** Demonstrates the Learning pillar — parameters are estimated purely from partial test observations and the DBN structure, without observing latent SEIR states directly.

In [ ]:
labels = ['beta (transmission)', 'sigma (E→I)', 'gamma (I→R)']
colors = ['#2980b9', '#8e44ad', '#16a085']

fig, ax = plt.subplots(figsize=(9, 4.5))
for idx, (label, color) in enumerate(zip(labels, colors)):
    ax.plot(history[:, idx], label=f'learned {label}', color=color, linewidth=2)
ax.set_xlabel('EM iteration')
ax.set_ylabel('Parameter value')
ax.set_title('Figure 4: EM Parameter Convergence (Real Data)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig4_em_convergence.png', dpi=150)
plt.show()

---
# Part 5 — Optional: Synthetic Data Comparison

The project description allows **simulated or real-world** data. Synthetic data is useful when we know ground-truth parameters and want to validate EM learning.

In [ ]:
from src.network import make_contact_network
from src.simulation import simulate_epidemic

syn_config = SimConfig(n_nodes=20, n_timesteps=40, test_probability=0.7, seed=42)
true_params = ModelParams(beta=0.30, sigma=0.20, gamma=0.10)
G_syn = make_contact_network(syn_config.n_nodes, syn_config.network_kind, syn_config.seed)
X_syn, Y_syn = simulate_epidemic(G_syn, true_params, syn_config)

learned_syn, hist_syn = em_learn(G_syn, Y_syn, ModelParams(0.1, 0.1, 0.1), n_iter=25, verbose=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for idx, (name, color) in enumerate(zip(['beta', 'sigma', 'gamma'], ['#2980b9', '#8e44ad', '#16a085'])):
    ax.plot(hist_syn[:, idx], color=color, linewidth=2, label=f'learned {name}')
    ax.axhline(true_params.as_array()[idx], color=color, linestyle=':', alpha=0.7, label=f'true {name}')
ax.set_xlabel('EM iteration')
ax.set_ylabel('Parameter value')
ax.set_title('Synthetic Data: EM Converges Toward True Parameters')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'True:    beta={true_params.beta}, sigma={true_params.sigma}, gamma={true_params.gamma}')
print(f'Learned: beta={learned_syn.beta:.3f}, sigma={learned_syn.sigma:.3f}, gamma={learned_syn.gamma:.3f}')

---
# Part 6 — Summary

## Figure reference for your report

| Figure | What it shows | PGM pillar |
|--------|---------------|------------|
| **Fig 0** | Contact network of individuals | Representation |
| **Fig 1** | SEIR epidemic curve over time | Data / Representation |
| **Fig 2** | P(infectious) heatmap — people × time | Inference |
| **Fig 3** | Network colored by max posterior P(I) | Inference |
| **Fig 4** | EM parameter convergence | Learning |

## Three PGM pillars — what this notebook demonstrated

1. **Representation** — Defined SEIR latent variables, 2-time-slice DBN structure, and CPTs on a real contact network
2. **Inference** — Forward–backward belief propagation to compute $P(X_i^t = I \mid Y)$
3. **Learning** — EM algorithm estimated β, σ, γ from partial real-world test observations

## Data used

- **Real-world:** Geneva COVID-19 contact tracing (GEgraph) — individual contacts + positive PCR tests
- **Optional synthetic:** Simulated SEIR epidemic for parameter validation

## Key takeaway

Even with **sparse, partial test results** on a **real outbreak subgraph**, the DBN infers who was likely infectious at each timestep and learns epidemiological parameters — fulfilling the project goal of modeling disease spread through a population with probabilistic graphical models.